# Project 2 — Panda Push from Pixels

Train an agent to **push the green cube onto the red target**, seeing the world **only through
pixels** and acting through **7 joint-angle deltas** — with nothing but a reward signal to learn from.

It looks simple; it isn't. The network has to learn, all at once: **perception** (tell the robot current configuration - joints, components and end-effector positions, the green cube and red target locations, ditsances, etc. in a raw image), **spatial geometry** from one fixed camera view, **inverse
kinematics** (how rotating joints moves the gripper in 3D), **contact physics** (friction, the cube's
inertia), and **action sequencing** (find → reach → push → settle). See the **README** for the full
write-up, the I/O contract, your levers, the tiered grading, and where to run.

Sections **0–2** below are provided: they walk through the environment, show what the agent actually
sees, and run a **naive baseline** that fails — build intuition here, then do your work under
**`## 3. Your solution`**.

> **Contract, in one line:** input `uint8 (B, 12, 112, 112)` in `[0, 255]` (4 stacked RGB frames — SB3's
> default `normalize_images=True` does the `/255`) → output `float32 (B, 7)` in `[-1, 1]`; the submitted
> `model.pt` is TorchScript, **≤ 4M params**, exported by `panda_push_pixels.export_model`. The full
> contract, the reward, and the grading rules are in the README.


## 0. Install

On Colab/Kaggle — run once per session.

In [ ]:
# ── Install (Colab / Kaggle / Paperspace) ──────────────────────────────────────
import subprocess, sys, os

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")

if ON_KAGGLE:
    import numpy as np
    if np.__version__ >= "2":
        # numpy 2.x is preinstalled on Kaggle but breaks pybullet/panda-gym.
        # Downgrade first, then restart the kernel — the ONLY reliable fix.
        _pip("numpy<2")
        print()
        print("=" * 65)
        print("  \u2705 numpy<2 installed.")
        print()
        print("  \u26a0\ufe0f  ACTION REQUIRED \u2014 restart the kernel now:")
        print("      Kaggle top menu \u2192 Run \u2192 Restart & Run All cells")
        print("      (this cell will be a no-op on the second run)")
        print("=" * 65)
        print()
        raise RuntimeError(
            "Kernel restart required after numpy downgrade — "
            "use Run → Restart & Run All cells"
        )
    # After restart numpy<2 is loaded; install the package normally.
    _pip("panda-push-pixels[train] @ git+https://github.com/kse-reinforcement-learning-2026-summer/panda-push-pixels.git@v11.0.0")

elif ON_COLAB:
    subprocess.run(["uv", "pip", "install", "--system", "-q", "numpy<2"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q",
        "panda-push-pixels[train] @ git+https://github.com/kse-reinforcement-learning-2026-summer/panda-push-pixels.git@v11.0.0"],
        check=True)

else:
    _pip("numpy<2")
    _pip("panda-push-pixels[train] @ git+https://github.com/kse-reinforcement-learning-2026-summer/panda-push-pixels.git@v11.0.0")

print("Install complete — panda_push_pixels ready.")

## 1. Environment Setup & Exploration

### 1.1 Initialize and inspect the environment

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import panda_push_pixels
from panda_push_pixels import contract

# Create the frozen evaluation environment
env = gym.make("PandaPushPixels-v0")

print("=== Environment Contract ===")
print(f"Observation space: {env.observation_space}")
print(f"  Expected shape: {contract.OBS_SHAPE}")
print(f"Action space: {env.action_space}")
print(f"  Expected dim: {contract.ACTION_DIM}")
print(f"Max episode steps: {contract.MAX_EPISODE_STEPS}")
print(f"Success distance threshold: {contract.DISTANCE_THRESHOLD} m")
print(f"Object/target size: {contract.OBJECT_SIZE} m")

# Reset and check observation
obs, info = env.reset(seed=42)
print(f"\nObservation: shape={obs.shape}, dtype={obs.dtype}")
print(f"  Range: [{obs.min():.3f}, {obs.max():.3f}]")
print(f"\nPrivileged info keys: {sorted(info.keys())}")
print(f"  object_position:   {info['object_position']}")
print(f"  target_position:   {info['target_position']}")
print(f"  ee_position:       {info['ee_position']}")
print(f"  object_to_target:  {info['object_to_target']:.4f} m")
print(f"  ee_to_object:      {info['ee_to_object']:.4f} m")
print(f"  is_touching:       {info['is_touching']}")
print(f"  is_success:        {info['is_success']}")

In [ ]:
import torch
device = 'mps' if torch.mps.is_available() else 'cpu'
from panda_push_pixels import PandaPushPixels
from panda_push_pixels import grading
from IPython.display import Video
from panda_push_pixels import render_episode, save_video
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from panda_push_pixels import extract_actor, grading
from panda_push_pixels import export_model, selfcheck

### 1.2 A closer look at the scene

A bigger render for a human-readable view — the agent's actual input is the small 112×112 stack
below, this is only for us to look at. The cube is green, the target is red.

In [ ]:
from panda_push_pixels import PandaPushPixels

# render_kwargs only changes what .render() returns for display -- it does NOT change the
# observation the agent receives (that stays frozen at 112x112, per the contract).
big_env = PandaPushPixels(render_kwargs=dict(render_width=360, render_height=360))
big_env.reset(seed=42)
plt.figure(figsize=(4, 4))
plt.imshow(big_env.render())
plt.title("cube (green) -> target (red)")
plt.axis("off")
plt.show()
big_env.close()

### 1.3 What the agent actually sees: 4 stacked RGB frames

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i, ax in enumerate(axes):
    # Frame i = 3 RGB channels [i*3 : i*3+3], transpose CHW -> HWC for imshow
    frame = obs[i*3:(i+1)*3].transpose(1, 2, 0)   # (112, 112, 3)
    ax.imshow(frame)                              # obs is uint8 [0,255] — imshow OK
    ax.set_title(f'Frame {i}', fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 1.4 Naive rollout — does the sparse reward give any learning signal?

A random policy ignores the observation entirely. Because the reward is sparse (`-1` every step,
`+50` only on success), almost every episode returns the same `-50` — there is no gradient
telling the agent it got "closer" to solving the task. This is the gap your reward shaping /
curriculum will need to close.

In [ ]:
import torch
from panda_push_pixels import grading

# Random policy: ignores the observation, samples uniformly from the action space.
# Wrapped as a callable obs_tensor -> action_tensor so grading.evaluate_policy can use it.
def random_policy(obs_tensor):
    return torch.as_tensor(env.action_space.sample(), dtype=torch.float32).unsqueeze(0)

print("Evaluating random policy...")
metrics = grading.evaluate_policy(random_policy, n_episodes=10, env=env)
print(f"Median return: {metrics['median_reward']:.1f}")
print(f"Mean return:   {metrics['mean_reward']:.1f} \u00b1 {metrics['std_reward']:.1f}")
print(f"Success rate:  {metrics['success_rate']:.2%}")
print(f"Range:         [{metrics['min_reward']:.0f}, {metrics['max_reward']:.0f}]")

### 1.5 Watch a random-policy episode

`render_episode` and `save_video` live in the package (`panda_push_pixels.viz`) — reuse them
instead of hand-rolling a recording loop in every notebook.

In [ ]:
from IPython.display import Video
from panda_push_pixels import render_episode, save_video

rollout = render_episode(env, policy=None, seed=42)   # policy=None -> uniform-random actions
print(f"Episode return: {rollout['episode_return']:.1f}")
print(f"Success: {rollout['success']}")

video_path = save_video(rollout["frames"], "random_episode.mp4", fps=20)
Video(video_path, width=300)

In [ ]:
env.close()

## 2. Naive baseline: PPO on the raw sparse reward

Let's just try it: plug the frozen env straight into PPO with near-default hyperparameters and
see what happens. No reward shaping, no curriculum, no custom CNN, no parallel envs — the naive
path, exactly like section 1.4's random policy but actually letting an algorithm try to learn.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

train_env = Monitor(gym.make("PandaPushPixels-v0"))

naive_model = PPO(
    "CnnPolicy", train_env,
    device="auto", verbose=1,
)
naive_model.learn(total_timesteps=10_000)
train_env.close()

### 2.2 Evaluate the naive policy

In [ ]:
from panda_push_pixels import extract_actor, grading

eval_env = gym.make("PandaPushPixels-v0")
naive_actor = extract_actor(naive_model)   # the actor only, no critic -- same object export_model would trace

print("Evaluating the naive PPO policy...")
naive_metrics = grading.evaluate_policy(naive_actor, n_episodes=10, env=eval_env)
print(f"Median return: {naive_metrics['median_reward']:.1f}")
print(f"Mean return:   {naive_metrics['mean_reward']:.1f} ± {naive_metrics['std_reward']:.1f}")
print(f"Success rate:  {naive_metrics['success_rate']:.2%}")
print(f"Range:         [{naive_metrics['min_reward']:.0f}, {naive_metrics['max_reward']:.0f}]")

### 2.3 Watch an episode of the naive policy

In [ ]:
rollout = render_episode(eval_env, policy=naive_actor, seed=42)
print(f"Episode return: {rollout['episode_return']:.1f}")
print(f"Success: {rollout['success']}")

video_path = save_video(rollout["frames"], "naive_ppo_episode.mp4", fps=20)
Video(video_path, width=300)

### 2.4 What gets submitted: export the actor to TorchScript

The grader never sees Stable-Baselines3 — only a standalone TorchScript `model.pt` containing
the **actor only** (no critic). `export_model` extracts it from the SB3 model and traces it;
`selfcheck` asserts the traced module reproduces `model.predict(deterministic=True)` exactly —
it verifies that the `/255` SB3's `normalize_images=True` (the default) applies is correctly
baked into the exported graph, so the standalone `model.pt` behaves identically on the raw uint8
observation the grader feeds it.

In [ ]:
from panda_push_pixels import export_model, selfcheck

model_path = export_model(naive_model, "model.pt")
max_err = selfcheck(naive_model, model_path)
print(f"Saved {model_path}")
print(f"selfcheck max error: {max_err:.2e}  (must be < 1e-4)")

### 2.5 Sanity-check against the grader's own pipeline

`grading.check_contract` and `grading.evaluate` are exactly what CI and the instructor's final
grading run against your `model.pt` — run them yourself before submitting.

In [ ]:
n_params = grading.check_contract(model_path)
print(f"Parameters: {n_params:,} (limit {contract.PARAM_LIMIT:,})")

final_metrics = grading.evaluate(model_path, n_episodes=10)
print(f"Success rate:  {final_metrics['success_rate']:.2%}  (pass threshold: {contract.SUCCESS_RATE_THRESHOLD:.0%})")
print(f"Median return: {final_metrics['median_reward']:.1f}  (diagnostic only -- not what's graded)")

eval_env.close()

---

## Why the naive baseline (probably) didn't work

With a sparse reward (`-1` every step, `+50` only once the cube has *dwelled* at the target for
5 consecutive steps or is still there at the time limit) and no shaping, almost every training
episode returns exactly `-50` regardless of what the policy did. PPO's advantage estimate is
computed *relative to the other rollouts in the same batch* — if every rollout gets the same
return, the advantage is ~0 everywhere, and there is no gradient telling the policy "this action
was better than that one." The policy has no reason to move towards the cube at all; it might as
well stay random. A few thousand steps on top of that is a tiny budget for a 112×112-pixel,
7-joint task — even a lucky early success has essentially no signal to consolidate into a
repeatable behavior.

## 3. Your solution

From here on it's your work. Sections 0-2 above are provided (environment walkthrough + a naive
baseline that fails); that naive baseline does **not** count toward your grade.

**Your levers** (full details and their hard limits are in the README): shape a dense **reward**
and/or a **curriculum** in a training-only `gymnasium.Wrapper` (read the privileged `info` dict);
pick and tune an **algorithm** — A2C, PPO, DDPG, TD3, SAC (Stable-Baselines3); build a **custom
CNN** within the 4M actor-parameter budget; and **scale** with parallel envs / a replay buffer
(mind your compute). These are levers, not a recipe — there is no single intended solution.

**To score, your code below must** import one of those SB3 algorithms and call `.learn(...)`
(your own run, not the naive baseline), then export your actor with
`panda_push_pixels.export_model(model, "model.pt")` and commit it.

**Grading tiers** (`tests/test.py`): **5** = trains an SB3 agent + a valid `model.pt` (loads, I/O
contract, ≤ 4M params); **10** = moves the cube in ≥ 80% of episodes; **15** = solves the push
(reaches the target and dwells) in ≥ 50% of episodes.

Run `pytest tests/test.py` locally before pushing (CI pushes are limited). It's a hard problem —
but a solvable one. Experiment, iterate, and lean on AI tools to brainstorm. Good luck!
